# 00 Setup
**Lab 2B, Look-Through.** Check your environment, your parameters and the test suite before you touch real data.

You start in **fixture mode**: a small synthetic release with fictional BDCs, in exactly the SEC's file layout. Notebook 02 switches you to your team's real release.

In [1]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "params.yaml").exists())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from lookthrough import paths, acquire, load, mapping, holdings, clean, reconcile, report, tables, pipeline
P = paths.load_params(ROOT)
RDIR = paths.release_dir(P, ROOT)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20)
print("release folder:", RDIR.relative_to(ROOT), "| fixture mode:", P["data"]["use_fixture"])

release folder: data\fixture\2026_08_bdc | fixture mode: True


In [2]:
import importlib.util, platform
print("Python", platform.python_version(), "| pandas", pd.__version__)
for m in ["pandas", "pyarrow", "requests", "yaml", "matplotlib"]:
    print(f"  {m:11s}", "ok" if importlib.util.find_spec(m) else "MISSING: pip install -r requirements.txt")
print("edgartools  ", "installed (used in notebook 07)" if importlib.util.find_spec("edgar") else "not installed (optional, notebook 07)")

Python 3.14.7 | pandas 3.1.0
  pandas      ok
  pyarrow     ok
  requests    ok
  yaml        ok
  matplotlib  ok
edgartools   not installed (optional, notebook 07)


In [3]:
print("Assigned release:", P["release"], "| tolerance:", P["reconcile"]["tolerance"])
if "Your Name" in P["sec"]["user_agent"]:
    print("ACTION: set sec.user_agent in params.yaml to your name and email before notebook 02")
print("Fixture present:", (ROOT / "data/fixture/2026_08_bdc/soi.tsv").exists())

Assigned release: 2026_08 | tolerance: 0.01
ACTION: set sec.user_agent in params.yaml to your name and email before notebook 02
Fixture present: True


In [5]:
import subprocess
r = subprocess.run([sys.executable, str(ROOT / "tests" / "run_tests.py")], capture_output=True, text=True, cwd=ROOT)
print(r.stdout[-1200:])
assert r.returncode == 0, "the test suite should pass on the fixture before you start"

  PASS  test_baseline_reconciliation_counts_every_filing
  PASS  test_checker_catches_a_silently_dropped_filing
  PASS  test_checker_passes_baseline_and_catches_tampering
  PASS  test_checker_requires_triage_categories
  PASS  test_columns_map_by_tag_and_usecols_cover_them
  PASS  test_current_period_comes_from_sub_when_soi_period_is_empty
  PASS  test_lean_num_keeps_custom_fair_value_facts
  PASS  test_manifest_is_portable_and_detects_changes
  PASS  test_norm_date_handles_three_formats
  PASS  test_parse_segments_readme_examples
  PASS  test_placeholder_user_agent_is_refused
  PASS  test_release_url_patterns
  PASS  test_soi_rows_keep_their_numbers_when_columns_are_limited
  PASS  test_split_rows_coalesce_to_one_row_per_holding
  PASS  test_superseded_filing_is_dropped
  PASS  test_total_rule_spares_ordinary_names
  PASS  test_unzip_handles_macos_entries_and_nesting
17/17 tests pass



**Your turn.** Set `sec.user_agent` and confirm your team's `release` in `params.yaml` (Case Study 2B, Section 4).